# Universidad Central · Big Data · S13
# ¿Cuándo y cómo nos ayuda Dask a ejecutar tareas distribuidas?

**Historia profesional:** recibimos títulos de procesos SECOP y necesitamos procesar lotes independientes en una máquina virtual. Primero ejecutamos una función local; después la ejecutamos con Dask Distributed y comprobamos **los mismos resultados, los workers reales y los cronómetros**. También estudiaremos MapReduce, la representación numérica TF-IDF y la exportación Parquet.

**Recursos de clase (dos):** [presentación y tutorial interactivo](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s1) y **este cuaderno**. Los scripts y Streamlit ya están preparados en el perfil S13 de Codespaces.

**Objetivo verificable:** conseguir dos JSON de una ejecución pequeña: procesamiento real de términos y espera externa simulada. Interpretar si Dask gana la tarea con clúster abierto y si compensa el costo de iniciarlo. **No se ejecutan 50.000 registros, ni Ollama, ni entrenamiento pesado.**

El bloque de clasificación UNSPSC queda al final como **ampliación opcional**, con ejecución desactivada por defecto. No es requisito para completar la sesión.


## Ruta principal · cinco unidades, una pregunta por unidad

| Unidad | Pregunta que responde | Acción del estudiante | Evidencia |
|---|---|---|---|
| 1. Arquitectura | ¿Qué diferencia hay entre CPU, núcleo, proceso, hilo, worker y nodo? | Consultar CPU y configurar 2 workers | Configuración real |
| 2. Repartir | ¿Cómo divido una tarea y combino resultados? | MapReduce y particiones | Suma/contador idénticos |
| 3. Dask real | ¿Cómo cambian las ejecuciones según la carga? | Procesamiento real SECOP y espera simulada | JSON por modo y dos cronómetros |
| 4. Representar texto | ¿Para qué sirven TF, DF, IDF, L2 y coseno? | Crear vectores en un minicorpus único | Tabla TF-IDF y ranking |
| 5. Transferir | ¿Cuándo distribuir y qué reutilizar en PySpark? | Exportar Parquet y decisión cerrada | Parquet, manifiesto y JSON |

**Regla de medición:** los valores de GitHub Actions fueron obtenidos en otro hardware, no representan el Codespace. Se reservan para documentación de profundización; aquí medimos un trabajo pequeño en el entorno del estudiante.

**La ruta principal termina antes de la ampliación opcional**. No necesitas evaluar modelos, calcular F1 ni instalar XGBoost para comprender Dask.


## Navegación directa y comienzo de la práctica

Las **24 diapositivas** se agrupan en cinco unidades; sus herramientas son parte de esta misma clase.

| Unidad | Presentación | En el cuaderno |
|---|---|---|
| 1. Arquitectura y recursos | [1–5](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s1) | Inspección del entorno y workers |
| 2. MapReduce y HDFS conceptual | [6–12](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s6) | Map, Reduce y comprobación pandas/Dask |
| 3. Texto como datos | [13–15](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s13) | TF-IDF manual, L2 y coseno |
| 4. Dask ejecutable | [16–20](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s16) | Caso real, espera simulada, recursos y tiempos |
| 5. Transferencia | [21–24](../Presentaciones/s13-de-una-maquina-a-un-cluster.html#s21) | Evidencia estructurada y Parquet |

**Abre el panel visual:**
```bash
streamlit run Laboratorios/S13_Codespaces/app_tfidf_unspsc.py --server.address 0.0.0.0 --server.port 8503
```

En Codespaces: **PORTS → 8503 → Private → Open in Browser**. La pestaña 5 permite lanzar la demostración pequeña y descargar el resultado de cada ejecución.

**Cómo se lee:** la presentación explica conceptos y permite predecir; este cuaderno ejecuta y contrasta. **Qué nos dice:** si los resultados coinciden y cuánto cuesta el trabajo. **Qué NO permite concluir:** que dos workers sean dos máquinas ni que una simulación equivalga a OCR real. **Error común:** comparar los tiempos históricos de 50.000 con un Codespace distinto.


## 0 · Decidir antes de instalar

**Pregunta profesional:** ¿qué problema justifica pagar el costo de coordinar trabajadores?

| Trabajo | Primera opción razonable |
|---|---|
| 200 registros que caben en memoria | pandas |
| Consulta SQL sobre muchos CSV/Parquet | considerar DuckDB |
| Muchas transformaciones independientes o datos mayores que memoria | Dask puede ayudar |
| Un modelo clásico con cientos de ejemplos | scikit-learn |
| Un único modelo profundo entrenado entre máquinas | algoritmo con soporte distribuido explícito |

**OJO:** Dask no es sinónimo de «más rápido». Dos workers dentro de un Codespace **no son dos computadores**. Cada herramienta resuelve una necesidad distinta.


## 1 · Crear tu entorno propio en el navegador

**HAZ ESTO AHORA.** 1. Abre [el repositorio](https://github.com/jazaineam1/BigData2026). 2. Pulsa **Code → Codespaces → ⋯ → New with options**. 3. En configuración del contenedor selecciona **S13 · Dask, MapReduce y modelos SECOP**. **No** elijas «Airflow Lab» (predeterminado) ni «S11 multimedia». 4. Crea un Codespace y abre una **Terminal → New Terminal**.

Dentro de la terminal ejecuta:

```bash
python --version
python -m pip show pandas
python -m pip show dask
python -m pip show scikit-learn
python Laboratorios/S13_Codespaces/verificar_entorno.py
```

Si falta una biblioteca, **dentro de Codespaces**, no en tu PC: `python -m pip install -r Laboratorios/S13_Codespaces/requirements.txt`. Abre este cuaderno en VS Code web y selecciona el kernel Python del Codespace. No necesita Ubuntu, Docker ni VS Code instalados localmente.

**Resultados:** debe terminar con `LISTO PARA S13`; si escogiste el perfil equivocado, crea otro Codespace con la configuración S13.


### 1A · Preparar rutas (por qué no importa si abriste el cuaderno desde Cuadernos)

`Path` maneja rutas; `sys.path` indica dónde encontrar las funciones reutilizables. Conserva visible el código: puedes adaptarlo a un proyecto ajeno.


In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd()
if not (RAIZ / "Datos").exists():
    RAIZ = RAIZ.parent

assert (RAIZ / "Datos/tc2_corpus_2026.json").exists(), "Abre el repositorio en Codespaces"
sys.path.insert(0, str(RAIZ / "Laboratorios/S13_Codespaces"))
print("Raíz del curso:", RAIZ)


### 1B · Comprobar biblioteca y archivos, no sólo que pip haya respondido

Esta celda ejecuta un cálculo mínimo Dask y comprueba que existen las dos fuentes del curso.


In [ ]:
import pandas as pd
import dask.dataframe as dd
from analisis import leer_corpus, leer_etiquetados

demo = pd.DataFrame({"grupo": ["A", "A", "B"], "valor": [10, 20, 30]})
mini = dd.from_pandas(demo, npartitions=2)
print(mini.groupby("grupo")["valor"].sum().compute())
assert int(mini.groupby("grupo")["valor"].sum().compute().loc["A"]) == 30
print("Fuentes y herramientas:", len(leer_corpus()), len(leer_etiquetados()))


## Laboratorio central · Dask ejecuta trabajo real con datos SECOP

Utilizaremos **hasta 1.000 títulos SECOP** en ocho lotes. La función por lote calcula frecuencias de términos y una huella SHA-256. El resultado se verifica de dos maneras: ejecuciones secuenciales y ejecución mediante workers reales de Dask.

Experimentaremos con dos modos **sobre la misma muestra**:

1. **`datos` — procesamiento real:** cuenta palabras y términos de los títulos. No espera artificialmente. Dask puede perder por el costo de coordinación: también es un aprendizaje válido.
2. **`espera` — concurrencia real con latencia simulada:** realiza el mismo conteo y además espera 0,6 segundos por lote para representar un servicio externo independiente. Dask coordina las tareas de verdad, pero **NO hace OCR ni ASR**.

Comparamos tres relojes distintos: tiempo secuencial de las tareas, Dask con workers preparados y Dask incluyendo el arranque. El script muestra por separado la preparación de entradas.


In [ ]:
import os
import subprocess
import sys
import json
from pathlib import Path

print("CPU lógicas del sistema:", os.cpu_count())
disponibles = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
print("CPU lógicas permitidas por afinidad:", disponibles)
from dask_cuando_si import cpus_disponibles
print("Límite operativo orientativo (afinidad + cuota cgroup):", cpus_disponibles())
print("Para inspeccionar físicamente la VM ejecuta en terminal: nproc; lscpu; free -h")
print("Una VM puede tener 1, 2 o 4 workers, cada uno con uno o varios hilos.")


In [ ]:
# Dos experimentos PEQUEÑOS de Dask real: datos SECOP y latencia externa simulada.
# Modifica SOLO WORKERS_DEMO_S13 para comparar 1, 2 o 4 workers; revisa nproc primero.
import json
import subprocess
import sys
from pathlib import Path
WORKERS_DEMO_S13 = 2
script_dask = RAIZ / "Laboratorios/S13_Codespaces/dask_cuando_si.py"
assert script_dask.is_file(), script_dask
mediciones_s13 = {}
for modo in ("datos", "espera"):
    run = subprocess.run(
        [sys.executable, str(script_dask), "--workers", str(WORKERS_DEMO_S13),
         "--tareas", "8", "--modo", modo, "--espera", "0.6"],
        check=True, capture_output=True, text=True, cwd=RAIZ, timeout=120,
    )
    prefijo = "RESULTADO_S13_DASK="
    lineas = [s[len(prefijo):] for s in run.stdout.splitlines()
              if s.startswith(prefijo)]
    assert len(lineas) == 1, "Se esperaba un resultado JSON por modo"
    salida = json.loads(lineas[0])
    assert salida["resultados_identicos"], "Dask no produjo el mismo resultado"
    assert len(salida["workers_reales"]) == WORKERS_DEMO_S13
    assert salida["palabras_procesadas"] > 0
    assert salida["vocabulario_unico"] > 0
    assert salida["palabras_procesadas"] >= sum(
        item["apariciones"] for item in salida.get("top_terminos", [])
    )
    mediciones_s13[modo] = salida
    print(f"\nMODO {modo.upper()} · {salida['tareas']} lotes · {salida['workers']} workers")
    print("Términos reales más frecuentes:", salida.get("top_terminos", [])[:3])
    print("Secuencial:", salida["segundos_secuencial"], "s")
    print("Dask caliente:", salida["segundos_dask_cluster_caliente"], "s")
    print("Dask (arranque + tareas):", salida["segundos_dask_total_incluido_arranque"], "s")
    print("Ganador caliente:", salida["ganador_caliente"],
          "| con arranque:", salida["ganador_total"])
print("\nAmbos modos comprobados; no se descargaron modelos ni se ejecutaron 50.000 fichas.")


**Cómo se lee:** en `datos` verás la frecuencia de palabras y el tiempo de transformar ocho particiones reales. En `espera` verás el mismo trabajo más la latencia externa simulada. `workers_reales` enumera procesos, hilos y límites de memoria.

**Qué nos dice:** si compensa distribuir tareas ligeras o tareas con esperas independientes en esa VM. Si el trabajo local tarda menos, Dask no deja de funcionar: su costo de coordinación domina.

**Qué NO permite concluir:** que el caso simulado haya ejecutado OCR, que todas las tareas escalen linealmente o que se hayan medido otros computadores. **Error común:** llamar «tiempo total del pipeline» al tiempo de Dask que excluye preparación de entrada y persistencia a disco.

**Prueba controlada:** cambia `WORKERS_DEMO_S13` de 2 a 4 únicamente si la VM habilita cuatro CPU lógicas y tiene RAM suficiente. El valor controla tanto la creación de workers como la comprobación de resultados.


## 2 · Sistemas de archivos: bloques, nodos y réplicas

Un sistema como **HDFS** divide archivos en bloques lógicos y almacena esos bloques en nodos. La replicación crea copias para tolerar fallos. El **NameNode** administra metadatos de ubicación; los **DataNodes** guardan bloques.

Esto NO es lo mismo que partir un DataFrame para procesarlo. La siguiente celda es una **SIMULACIÓN en Python**, no un clúster Hadoop real.

| Elemento | Función |
|---|---|
| Nodo | máquina o proceso que participa |
| Bloque | unidad de almacenamiento |
| Réplica | otra copia de un mismo bloque |
| Partición | subconjunto lógico de trabajo |
| Scheduler | coordinador de tareas |
| Worker | ejecutor de tareas |

**Decisión:** ¿qué sucede si se pierde un nodo que guarda la única copia de un bloque?


In [ ]:
ubicaciones = {
    "bloque_1": ["nodo_A", "nodo_B"],
    "bloque_2": ["nodo_B", "nodo_C"],
    "bloque_3": ["nodo_A", "nodo_C"],
}
nodo_caido = "nodo_B"

for bloque, nodos in ubicaciones.items():
    disponibles = [n for n in nodos if n != nodo_caido]
    print(bloque, "→", disponibles)
    assert disponibles, f"Bloque no recuperable: {bloque}"


**Cómo se lee:** cada bloque conserva una ubicación distinta de nodo_B.  
**Qué nos dice:** copiar bloques en distintos nodos puede proteger contra una falla simple.  
**Qué NO permite concluir:** que un Codespace sea un HDFS real ni que todas las políticas de replicación garanticen disponibilidad.  
**Error común:** confundir replicar los datos con paralelizar el cálculo.


## 3 · MapReduce: la trampa del promedio de promedios

Imagina 3 lotes de contrataciones con diferentes tamaños. En el primer lote hay 2 importes, en el segundo 5 y en el tercero 100. **Promediar las tres medias de los lotes es incorrecto** porque no representan el mismo número de observaciones.

`map` produce estadísticas parciales. `reduce` combina parciales sin perder los pesos. **Es el modelo MapReduce implementado en Python**, no un servicio de Apache Hadoop.


In [ ]:
lotes = [
    [100, 120],
    [10, 12, 11, 9, 8],
    [1] * 100,
]
parciales = [(sum(lote), len(lote)) for lote in lotes]
print("MAP (suma, cantidad):", parciales)


In [ ]:
suma_total = sum(suma for suma, cantidad in parciales)
cantidad_total = sum(cantidad for suma, cantidad in parciales)

promedio_correcto = suma_total / cantidad_total
promedio_incorrecto = sum(s / n for s, n in parciales) / len(parciales)

print("REDUCE correcto:", round(promedio_correcto, 3))
print("Promedio de promedios:", round(promedio_incorrecto, 3))
assert round(promedio_correcto, 6) == round(370/107, 6)


**Cómo se lee:** 370 unidades en 107 observaciones dan una media de aproximadamente 3,458.  
**Qué nos dice:** para un promedio distribuido necesitamos **suma y cantidad**, no sólo medias locales.  
**Qué NO permite concluir:** que ya usamos Hadoop MapReduce ni que un algoritmo cualquiera sea divisible así.  
**Error común:** `mean(mean(lote))` ignora los tamaños de las particiones.

**Transfiere:** esta estrategia sirve para métricas de ventas, llamadas o tiempos por cliente, siempre que conserves los pesos.


## Datos SECOP suministrados para operaciones distribuidas

En este laboratorio los datos ya están preparados. La muestra permite practicar lectura, partición, agrupación y exportación de Parquet. En esta clase **no corresponde evaluar la calidad administrativa de los registros**.

La columna `modalidad_de_contratacion` se usa aquí **solo para agrupar y contar**, nunca como segunda tarea de clasificación. La ampliación opcional al final utiliza un único objetivo supervisado: el segmento UNSPSC declarado.


In [ ]:
tabla = leer_etiquetados()
print("Dimensiones:", tabla.shape)
print(tabla["modalidad_de_contratacion"].value_counts().head(7))
print(tabla[["id_del_proceso", "descripci_n_del_procedimiento"]].head(2))
assert tabla["id_del_proceso"].is_unique


## 5 · pandas primero: la referencia de corrección

`groupby` agrupa filas por modalidad y `size()` cuenta filas. Esta salida es la referencia contra la cual compararemos Dask.


In [ ]:
conteo_pandas = tabla.groupby("modalidad_de_contratacion").size().sort_index()
print(conteo_pandas.head(10))
print("Total:", int(conteo_pandas.sum()))
assert conteo_pandas.sum() == 1000


## 6 · Dask DataFrame: el mismo cálculo por particiones

`from_pandas` divide el DataFrame en particiones. `groupby` construye operaciones diferidas; `compute()` materializa el resultado. Una partición **no** es un servidor.

¿Cuál sería el costo de Dask en sólo mil registros? El coordinador, el grafo de tareas y la comunicación pueden tardar más que la operación real.


In [ ]:
dask_tabla = dd.from_pandas(tabla, npartitions=4)
pendiente = dask_tabla.groupby("modalidad_de_contratacion").size()

print("Particiones:", dask_tabla.npartitions)
print("Antes de compute:", type(pendiente).__name__)
conteo_dask = pendiente.compute().sort_index()
valores_pandas = {str(k): int(v) for k, v in conteo_pandas.items()}
valores_dask = {str(k): int(v) for k, v in conteo_dask.items()}
assert valores_pandas == valores_dask
print("Resultado Dask:", conteo_dask.head(), "\nCoincide: SÍ")


### 6A · Medir sin inventar «Dask siempre gana»

Una medición debe incluir el tiempo de ejecución, el número de particiones y el tamaño del conjunto. Se repite en condiciones comparables; el primer tiempo puede incluir inicialización, cachés y procesos. Comparamos por **corrección** antes de velocidad.


In [ ]:
from analisis import comparar_pandas_dask
mediciones = []
for n_particiones in (1, 2, 4):
    mediciones.append(comparar_pandas_dask(tabla, npartitions=n_particiones))

display(pd.DataFrame(mediciones)[
    ["filas", "particiones", "segundos_pandas", "segundos_dask", "coinciden"]
])


**Cómo se lee:** los segundos se miden en TU Codespace y pueden cambiar en cada ejecución.  
**Qué nos dice:** la comparación es local, sobre el mismo conjunto de 1.000 registros.  
**Qué NO permite concluir:** cuál herramienta es más rápida para millones de filas en otro sistema.  
**Error común:** confundir aumentar particiones con aumentar máquinas o comparar ejecuciones con cargas distintas.


## ¿Por qué convertir palabras en vectores TF-IDF?

El texto no es directamente una matriz numérica. TF-IDF convierte cada documento en pesos que permiten **buscar documentos relacionados** mediante coseno y, si se necesita en una ampliación, alimentar un clasificador de texto.

Este mismo minicorpus se usa **sin cambios en la presentación, el cuaderno y Streamlit**:

- **A:** «mantenimiento computadores»
- **B:** «infraestructura computadores»
- **C:** «hospital mantenimiento»

**TF(A):** apariciones del término en A. **DF:** número de documentos que contienen el término. **IDF suavizado:** `ln((1+N)/(1+DF))+1`, con N=3. **TF×IDF(A):** peso sin normalizar. **L2(A):** peso después de dividir por la longitud euclídea del vector A.

Por ejemplo, `hospital` solo aparece en C, por eso IDF=1,6931. En A tiene TF=0 y peso final 0. `computadores` y `mantenimiento` tienen DF=2, IDF=1,2877 y peso L2=0,7071 cada una en A.

**¿Dónde entra Dask?** Puede repartir la transformación de muchas particiones de documentos cuando el volumen justifica el costo. No modifica la fórmula TF-IDF.


### Herramienta: calcula y explica cada columna

**Objetivo:** construir una tabla de seis columnas para el documento A: término, DF, IDF, TF(A), TF×IDF(A) y L2(A). Después modifica un texto y observa qué cambia. El ejemplo queda idéntico al de la presentación y al del panel Streamlit.

**Cómo se lee:** DF depende de los tres documentos; TF(A) y L2(A) dependen del documento A. Un término puede tener IDF alto y peso cero en A si no aparece allí.

**Qué nos dice:** el documento puede convertirse en un vector numérico para coseno. **Qué NO permite concluir:** que compartir términos equivalga a comprender el significado. **Error común:** comparar TF×IDF sin normalización con L2 como si fueran columnas de la misma escala.


In [ ]:
from collections import Counter
import math
import pandas as pd

mini_documentos = [
    "mantenimiento computadores",   # A
    "infraestructura computadores", # B
    "hospital mantenimiento",       # C
]
tokens = [doc.lower().split() for doc in mini_documentos]
vocabulario = sorted(set(termino for doc in tokens for termino in doc))
N = len(tokens)
pesos_sin_normalizar = {}
for termino in vocabulario:
    df = sum(termino in doc for doc in tokens)
    idf = math.log((1 + N) / (1 + df)) + 1
    tf_a = tokens[0].count(termino)
    pesos_sin_normalizar[termino] = {
        "término": termino,
        "DF": df,
        "IDF": idf,
        "TF(A)": tf_a,
        "TF×IDF(A)": tf_a * idf,
    }
norma_a = math.sqrt(sum(fila["TF×IDF(A)"] ** 2
                        for fila in pesos_sin_normalizar.values()))
tabla_tfidf_a = pd.DataFrame([
    {**fila, "L2(A)": fila["TF×IDF(A)"] / norma_a if norma_a else 0.0}
    for fila in pesos_sin_normalizar.values()
])
display(tabla_tfidf_a.round(4))
print("Norma antes:", round(norma_a,4),
      "| norma después:", round(math.sqrt((tabla_tfidf_a["L2(A)"]**2).sum()),4))
assert len(vocabulario) == 4
assert int(tabla_tfidf_a.loc[tabla_tfidf_a["término"] == "hospital", "DF"].iloc[0]) == 1
assert abs(float(tabla_tfidf_a.loc[tabla_tfidf_a["término"] == "hospital", "IDF"].iloc[0]) - (math.log(2)+1)) < 1e-8
assert abs(float(tabla_tfidf_a.loc[tabla_tfidf_a["término"] == "computadores", "L2(A)"].iloc[0]) - 2**-0.5) < 1e-8


**Cómo se lee:** cada fila muestra el mismo término en el vocabulario del corpus. `hospital` tiene DF=1 e IDF≈1,6931, pero su peso L2 en A vale 0. `mantenimiento` y `computadores` tienen TF(A)=1 y pesos L2≈0,7071.

**Qué nos dice:** el vector del documento A está normalizado: la raíz de la suma de cuadrados de sus valores finales es 1. **Qué NO permite concluir:** que una palabra con mayor IDF siempre influya en todos los documentos. **Error común:** cambiar un término en B y esperar que TF(A) cambie automáticamente.


### Estación 2 · Contrastar la fórmula manual con scikit-learn

`TfidfVectorizer` primero construye un **vocabulario**, luego calcula la frecuencia documental, IDF, TF ponderado y normaliza cada fila. Usaremos `min_df=1` sólo para el ejemplo de tres documentos. En el benchmark grande `min_df=2` es razonable porque hay suficientes documentos.

**Mini ficha de funciones:**

| Función | Parámetros y devolución | Error frecuente |
|---|---|---|
| `fit()` | Aprende vocabulario e IDF del corpus de entrenamiento | Aplicarla sobre el test |
| `transform()` | Devuelve matriz dispersa de `filas × términos` | Interpretar un cero como dato faltante |
| `fit_transform()` | Hace ambas operaciones en entrenamiento | Repetirla sobre el test, cambiando columnas |
| `get_feature_names_out()` | Nombres de términos en orden de columnas | Suponer que cada vector es una etiqueta |


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

toy = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b",
                      min_df=1, norm="l2",
                      smooth_idf=True, sublinear_tf=False)
mini_X = toy.fit_transform(mini_documentos)
mini_tabla = pd.DataFrame(
    np.round(mini_X.toarray(), 5),
    index=["A", "B", "C"],
    columns=toy.get_feature_names_out(),
)
display(mini_tabla)
assert mini_X.shape == (3, 4)
assert np.allclose(np.linalg.norm(mini_X.toarray(), axis=1), 1.0)
assert np.allclose(mini_X.toarray()[0], tabla_tfidf_a.set_index("término").loc[
    toy.get_feature_names_out(), "L2(A)"
].to_numpy())


**Cómo se lee:** cada fila es el vector de un texto; cada columna es una palabra. Los valores ya están normalizados, por eso no son iguales a los productos TF×IDF sin normalizar de la celda anterior.  
**Qué nos dice:** podemos reproducir la aritmética y comprobar matemáticamente lo que entrega la biblioteca.  
**Qué NO permite concluir todavía:** que el algoritmo entiende contexto semántico o calidad del contrato.  
**Error común:** creer que los pesos de TF-IDF son probabilidades de cada categoría.

**Transferencia:** prueba `norm=None`, compara la matriz y explica por qué cambió.


### Estación 3 · Similitud coseno: buscar un expediente sin entrenar un clasificador

TF-IDF permite **recuperar textos relacionados** incluso sin etiquetas. La consulta y los documentos deben representarse con el mismo vocabulario. La similitud coseno mide su alineación:

\[
\cos(\theta) = \frac{a\cdot b}{\|a\|\,\|b\|}
\]

Con vectores normalizados L2, equivale al producto punto. En un corpus de TF-IDF no negativo suele estar entre 0 y 1; un 0 puede significar palabras distintas, no temas necesariamente diferentes.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

consulta = "quiero mantenimiento de computadores"
consulta_vector = toy.transform([consulta])
similares = cosine_similarity(consulta_vector, mini_X).ravel()
tabla_similitud = pd.DataFrame({
    "documento": ["A", "B", "C"],
    "texto": mini_documentos,
    "similitud_coseno": np.round(similares, 4),
}).sort_values("similitud_coseno", ascending=False)
display(tabla_similitud)
assert len(similares) == 3


**Cómo se lee:** el documento con coseno más alto comparte términos relevantes con la consulta.  
**Qué nos dice:** TF-IDF habilita una búsqueda léxica reproducible sin llamadas a modelos grandes.  
**Qué NO permite concluir todavía:** que frases con sinónimos no presentes sean irrelevantes.  
**Error común:** construir otro `TfidfVectorizer` para la consulta; los índices de columnas ya no coincidirían.

**Reto:** reemplaza «computadores» por «PC» y observa si la similitud cae. Esa limitación motiva probar EmbeddingGemma 2.


## 10 · Guardar evidencia reutilizable con Parquet

Parquet es un formato **columnar**. El archivo siguiente mantiene procesos únicos, modalidad y descripción; no guarda por sí mismo los pesos de un modelo entrenado. La sesión 14 lo leerá con PySpark.


In [ ]:
from analisis import escribir_resultados
manifest = escribir_resultados(tabla)
print("Salida:", manifest["salida"])
print("Filas:", manifest["filas"])
print("SHA-256 fuente:", manifest["sha256_fuente"][:20], "...")

archivo = RAIZ / "Laboratorios/S13_Codespaces/salidas/secop_s13.parquet"
reabierto = pd.read_parquet(archivo)
assert reabierto["id_del_proceso"].is_unique
print("Reabierto:", reabierto.shape)


**Cómo se lee:** se exportaron 1.000 procesos únicos de la muestra y se recuperaron desde Parquet.  
**Qué nos dice:** la entrega es tabular, trazable a una fuente específica y legible.  
**Qué NO permite concluir:** que el archivo Parquet tenga los modelos entrenados o preserve una licencia externa no registrada.  
**Error común:** confundir igualdad lógica de filas con igualdad byte a byte de dos archivos Parquet reescritos.


## Cierre del laboratorio principal · decisión basada en tu ejecución

El ejercicio se completa con los dos modos del laboratorio (`datos` y `espera`) y una decisión estructurada. Antes de decidir, verifica `resultados_identicos=True`, los workers y los tiempos del **mismo Codespace**.

**Criterios formativos (no modifican el PDA):**

| Criterio | Evidencia verificable |
|---|---|
| Recursos | CPU lógicas disponibles y número real de workers |
| Corrección | Resultados secuenciales y Dask idénticos |
| Reparto | Direcciones de workers y número de tareas |
| Interpretación | Diferencia entre Dask caliente y Dask con arranque |
| Transferencia | Parquet, manifiesto y una arquitectura elegida de un menú cerrado |

El modelo de clasificación UNSPSC y sus gráficos se conservan después del cierre como ampliación opcional, sin ejecutarse en `Run All`.


In [ ]:
# Decisión final de Dask: elige los índices de dos menús cerrados.
# No corresponde redactar un ensayo ni entrenar un modelo.
# 1 = secuencial para trabajos muy cortos
# 2 = Dask 2 workers para varios trabajos independientes con espera
# 3 = medir 4 workers si la VM habilita CPU y RAM suficientes
ARQUITECTURA = None
LIMITE = None

OPCIONES_ARQUITECTURA = {
    1: "Secuencial cuando la coordinación domina",
    2: "Dask 2 workers para tareas independientes con espera",
    3: "Comparar 4 workers con recursos suficientes y las mismas tareas",
}
OPCIONES_LIMITE = {
    1: "El tiempo de arranque puede superar la ganancia",
    2: "La latencia de OCR/ASR fue SIMULADA, no un servicio real",
    3: "La VM comparte CPU y RAM entre los workers",
    4: "El cronómetro excluye algunas operaciones de entrada y persistencia",
}
print("Arquitectura:", OPCIONES_ARQUITECTURA)
print("Límite:", OPCIONES_LIMITE)
if ARQUITECTURA is None or LIMITE is None:
    print("Elige ambos índices y vuelve a ejecutar la celda para descargar evidencia.")
else:
    if type(ARQUITECTURA) is not int or ARQUITECTURA not in OPCIONES_ARQUITECTURA:
        raise ValueError("Arquitectura: 1, 2 o 3")
    if type(LIMITE) is not int or LIMITE not in OPCIONES_LIMITE:
        raise ValueError("Límite: 1, 2, 3 o 4")
    import json
    salida_evidencia = {
        "sesion": 13,
        "arquitectura": OPCIONES_ARQUITECTURA[ARQUITECTURA],
        "arquitectura_codigo": ARQUITECTURA,
        "limite": OPCIONES_LIMITE[LIMITE],
        "limite_codigo": LIMITE,
        "mediciones": {
            modo: {
                "workers": dato["workers"],
                "resultados_identicos": dato["resultados_identicos"],
                "segundos_secuencial": dato["segundos_secuencial"],
                "segundos_dask_cluster_caliente": dato["segundos_dask_cluster_caliente"],
                "segundos_dask_total_incluido_arranque": dato["segundos_dask_total_incluido_arranque"],
            }
            for modo, dato in mediciones_s13.items()
        },
    }
    destino = RAIZ / "Laboratorios/S13_Codespaces/salidas/decision_s13.json"
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(json.dumps(salida_evidencia, indent=2, ensure_ascii=False),
                       encoding="utf-8")
    print("Evidencia:", destino)


## Puente a S14 · ¡Dask está funcionando!

En S13 comprobaste cómo una sola VM puede ejecutar varias tareas sobre workers, cómo se agrupan resultados mediante MapReduce y cómo se convierten palabras en vectores con TF-IDF. Ahora existe evidencia propia de tu Codespace, no un benchmark copiado.

**Producto mínimo:** dos modos ejecutados con resultados idénticos, JSON de workers/tiempos, archivo Parquet y una decisión cerrada con sus límites.

**En S14:** utilizaremos PySpark para procesar el mismo Parquet y comparar un nuevo motor distribuido.

**Profundización no obligatoria:** el bloque siguiente demuestra cómo un clasificador LinearSVC utiliza vectores TF-IDF para sugerir segmentos UNSPSC. Está desactivado por defecto y no forma parte de la evidencia de Dask.


## Ampliación opcional · clasificación de textos SECOP por segmento UNSPSC

Esta ampliación usa **un solo objetivo de clasificación**: predecir el segmento UNSPSC declarado a partir del título y la descripción del procedimiento. La agrupación por modalidad anterior fue un ejemplo de DataFrame, no una segunda clasificación.

**Por defecto no se ejecuta en Run All.** Para realizarla de manera voluntaria, cambia `EJECUTAR_AMPLIACION_S13` a `True` en la siguiente celda y ejecuta las celdas de este bloque en orden. Solo utiliza la muestra pequeña versionada; no activa 50.000 registros, Ollama ni modelos externos.


In [ ]:
EJECUTAR_AMPLIACION_S13 = False  # Sólo si se solicita la ampliación opcional.
if globals().get('EJECUTAR_AMPLIACION_S13', False):
    import re
    from pathlib import Path

    archivo_secop = RAIZ / "Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv"
    columnas_secop = [
        "id_del_proceso", "nombre_del_procedimiento",
        "descripci_n_del_procedimiento", "codigo_principal_de_categoria",
        "fecha_de_publicacion_del",
    ]
    secop_pequeno = pd.read_csv(
        archivo_secop, usecols=columnas_secop,
        dtype=str, keep_default_na=False,
    ).drop_duplicates("id_del_proceso")
    patron = re.compile(r"^(?:V[0-9]+\.)?([0-9]{8})$")
    def segmento_unspsc(codigo):
        m = patron.fullmatch(str(codigo).strip())
        return m.group(1)[:2] if m else None

    secop_pequeno["segmento"] = secop_pequeno[
        "codigo_principal_de_categoria"].map(segmento_unspsc)
    secop_pequeno["texto"] = (
        secop_pequeno["nombre_del_procedimiento"] + " " +
        secop_pequeno["descripci_n_del_procedimiento"]
    ).str.strip()
    print("Procesos únicos:", len(secop_pequeno))
    print("Sin segmento válido:", secop_pequeno["segmento"].isna().sum())
    display(secop_pequeno[["id_del_proceso", "segmento", "texto"]].head(5))


**Cómo se lee:** cada fila es una ficha de proceso; `texto` será la entrada para TF-IDF y `segmento` una categoría administrativa de demostración.  
**Qué nos dice:** podemos procesar un mismo esquema por partes y unir resultados.  
**Qué NO permite concluir:** que este registro contenga un dictamen jurídico ni que sea una prueba de fraude.  
**Error común:** convertir en objetivo del laboratorio una auditoría de datos que no es parte de la enseñanza de Dask.

**Continuamos:** veremos cómo repartir transformaciones y entrenamientos; no se solicita al estudiante elaborar un informe de calidad de datos.


### Clasificador opcional: TF-IDF + LinearSVC

Solo si ejecutaste la ampliación: el texto es la entrada y el **segmento UNSPSC declarado** es la etiqueta. `fit` se realiza con textos de entrenamiento, `predict` sobre fichas posteriores y se compara con un baseline mayoritario. No se infiere fraude ni legalidad.

Este bloque no hace falta para completar el objetivo principal de Dask.


In [ ]:
if globals().get('EJECUTAR_AMPLIACION_S13', False):
    from sklearn.svm import LinearSVC
    from sklearn.dummy import DummyClassifier
    from sklearn.metrics import f1_score
    from sklearn.pipeline import make_pipeline

    clases_objetivo = {"80", "85", "93", "81", "72"}
    ejemplos = secop_pequeno[
        secop_pequeno["segmento"].notna()
        & secop_pequeno["texto"].str.len().ge(20)
    ].copy()
    ejemplos["clase"] = ejemplos["segmento"].where(
        ejemplos["segmento"].isin(clases_objetivo), "OTRAS")
    ejemplos["fecha"] = pd.to_datetime(
        ejemplos["fecha_de_publicacion_del"], errors="coerce")
    ejemplos = ejemplos.dropna(subset=["fecha"]).sort_values(
        ["fecha", "id_del_proceso"])
    i = int(.8 * len(ejemplos))
    train_secop, test_secop = ejemplos.iloc[:i], ejemplos.iloc[i:]
    modelito = make_pipeline(
        TfidfVectorizer(max_features=3000, min_df=2, ngram_range=(1,2)),
        LinearSVC(class_weight="balanced", random_state=42),
    )
    modelito.fit(train_secop["texto"], train_secop["clase"])
    predicciones = modelito.predict(test_secop["texto"])
    control = DummyClassifier(strategy="most_frequent")
    control.fit(np.zeros((len(train_secop),1)), train_secop["clase"])
    base_pred = control.predict(np.zeros((len(test_secop),1)))
    etiquetas_fijas = ["72","80","81","85","93","OTRAS"]
    print("Train:",len(train_secop),"Test:",len(test_secop))
    print("F1 macro modelo:", round(f1_score(
        test_secop["clase"], predicciones,
        labels=etiquetas_fijas, average="macro", zero_division=0),4))
    print("F1 macro mayoritaria:", round(f1_score(
        test_secop["clase"], base_pred,
        labels=etiquetas_fijas, average="macro", zero_division=0),4))


### Gráficos opcionales: comparar unigramas y bigramas en el mismo test

Si activaste la ampliación, la siguiente celda genera dos figuras PNG y una tabla CSV para explorar cómo cambia la clasificación. No debe tratarse como un segundo benchmark distribuido ni repetirse sobre grandes conjuntos.


In [ ]:
if globals().get('EJECUTAR_AMPLIACION_S13', False):
    # Dos gráficos y tabla trazable, sobre SOLO la muestra pequeña (sin jobs pesados).
    import hashlib
    import matplotlib.pyplot as plt
    from sklearn.metrics import classification_report
    from sklearn.pipeline import make_pipeline
    from sklearn.svm import LinearSVC
    from sklearn.feature_extraction.text import TfidfVectorizer

    nombres = ["Unigramas", "Unigramas + bigramas"]
    configs = [(1, 1), (1, 2)]
    resultados_ngramas = []
    predicciones_por_config = {}
    for nombre, ngramas in zip(nombres, configs):
        pipeline = make_pipeline(
            TfidfVectorizer(max_features=3000, min_df=2, ngram_range=ngramas),
            LinearSVC(class_weight="balanced", random_state=42),
        )
        pipeline.fit(train_secop["texto"], train_secop["clase"])
        pred = pipeline.predict(test_secop["texto"])
        met = classification_report(
            test_secop["clase"], pred, labels=etiquetas_fijas,
            zero_division=0, output_dict=True,
        )
        predicciones_por_config[nombre] = met
        resultados_ngramas.append({
            "configuracion": nombre,
            "ngramas": str(ngramas),
            "f1_macro": float(met["macro avg"]["f1-score"]),
            "n_train": int(len(train_secop)),
            "n_test": int(len(test_secop)),
        })

    tabla_ngramas = pd.DataFrame(resultados_ngramas)
    test_hash = hashlib.sha256(
        "\n".join(test_secop["id_del_proceso"].astype(str)).encode("utf-8")
    ).hexdigest()
    tabla_ngramas["test_sha256_ids"] = test_hash
    carpeta_evidencia = RAIZ / "Laboratorios/S13_Codespaces/salidas"
    carpeta_evidencia.mkdir(parents=True, exist_ok=True)
    tabla_ngramas.to_csv(carpeta_evidencia / "s13_ngramas_mismo_test.csv", index=False)
    display(tabla_ngramas)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(tabla_ngramas["configuracion"], tabla_ngramas["f1_macro"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1 macro (mismo test)")
    ax.set_title("A · Cambiar n-gramas sin cambiar prueba")
    fig.tight_layout()
    fig.savefig(carpeta_evidencia / "s13_A_f1_ngramas.png", dpi=125)
    plt.show()

    reporte = predicciones_por_config["Unigramas + bigramas"]
    f1_clase = pd.Series({c: reporte[c]["f1-score"] for c in etiquetas_fijas})
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(f1_clase.index, f1_clase.values)
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1 por segmento")
    ax.set_xlabel("Segmento UNSPSC declarado")
    ax.set_title("B · Desempeño desigual entre clases")
    fig.tight_layout()
    fig.savefig(carpeta_evidencia / "s13_B_f1_por_clase.png", dpi=125)
    plt.show()
    print("Mismo test SHA-256:", test_hash)
    print("Evidencia lista:", carpeta_evidencia)


**Cómo se lee (solo ampliación):** el gráfico A compara F1 macro con el mismo conjunto de prueba; el gráfico B muestra resultados por segmento. **Qué nos dice:** los pesos TF-IDF pueden alimentar un modelo supervisado sencillo. **Qué NO permite concluir:** que el segmento asignado pruebe una característica jurídica del proceso. **Error común:** comparar entrenamientos con pruebas diferentes.


**Cómo se lee:** F1 macro cercano a 1 sería mejor; en este conjunto pequeño importa compararlo con el control y revisar la F1 de las categorías minoritarias.  
**Qué nos dice:** si el texto aporta señales para aproximar la etiqueta administrativa.  
**Qué NO permite concluir todavía:** que el sistema generalice a SECOP completo ni que esté validado para hacer revisiones jurídicas.  
**Error común:** reportar sólo exactitud general cuando el segmento 80 domina.

**Reto:** ajusta sólo `ngram_range=(1,1)` y compara la calidad sobre el **mismo test**. No ajustes los hiperparámetros observando repetidamente el test final.
